# Heart Health Information Assistant

A teaching prototype that classifies heart-health questions, retrieves general educational information, and applies safety rules. It is **not a diagnostic or treatment tool**. Emergency warning signs take priority over model predictions.

Run top to bottom in Google Colab. The notebook uses scikit-learn and NLTK, trains on a small illustrative corpus, reports evaluation metrics, and ends with an interactive multi-query widget plus required scenario checks.

In [ ]:
# Colab setup
!pip -q install scikit-learn nltk ipywidgets
import re, json, warnings
from collections import Counter
import nltk
for resource in ("wordnet", "omw-1.4"):
    try: nltk.download(resource, quiet=True)
    except Exception: pass
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay)
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Markdown


## 1. Hierarchical taxonomy

Four top-level branches each contain two subcategories. The leaf labels below are the classifier intents.

In [ ]:
TAXONOMY = {
  "Symptoms and urgent concerns": {"chest_pain": "Chest discomfort questions", "palpitations": "Fast or irregular heartbeat"},
  "Risk and prevention": {"risk_factors": "Risk factors", "prevention": "Prevention and healthy habits"},
  "Conditions and care": {"blood_pressure": "Blood pressure", "heart_failure": "Heart failure education"},
  "Tests and treatment": {"tests": "Heart tests", "medications": "Medication education"},
}
INTENTS = [leaf for branch in TAXONOMY.values() for leaf in branch]
print("Intents:", INTENTS)


## 2. Labeled corpus (64 example queries)

Eight examples per leaf intent. This compact, hand-authored corpus is for demonstration only; real deployments need clinically reviewed, larger, representative data and independent validation.

In [ ]:
EXAMPLES = {
"chest_pain": ["What can cause chest discomfort?", "I feel pressure in my chest", "Why does my chest hurt when I breathe?", "Can heartburn feel like chest pain?", "What should I know about chest tightness?", "My chest feels sore after exercise", "Does anxiety cause chest discomfort?", "How is chest pain evaluated?"],
"palpitations": ["Why does my heart sometimes race?", "What are heart palpitations?", "My heartbeat feels irregular", "Can caffeine cause a pounding heart?", "What causes skipped beats?", "How are palpitations checked?", "Is a fluttering heartbeat common?", "My heart beats fast at rest, why?"],
"risk_factors": ["What increases the risk of heart disease?", "Does family history affect heart health?", "Can smoking harm my heart?", "Is diabetes a heart disease risk?", "How does cholesterol affect the heart?", "Does age change cardiovascular risk?", "Can stress raise heart risk?", "What are common heart attack risk factors?"],
"prevention": ["How can I keep my heart healthy?", "What exercise supports heart health?", "What foods are good for the heart?", "How much activity is recommended for heart wellness?", "How can I reduce my heart disease risk?", "Are fruits and vegetables heart healthy?", "What lifestyle changes protect the heart?", "How does quitting smoking help my heart?"],
"blood_pressure": ["What does high blood pressure mean?", "How is blood pressure measured?", "Why is hypertension harmful?", "What is a normal blood pressure reading?", "Can lifestyle changes lower blood pressure?", "What do the two blood pressure numbers mean?", "How often should blood pressure be checked?", "What is hypertension?"],
"heart_failure": ["What is heart failure?", "What symptoms can heart failure cause?", "Does heart failure mean the heart stopped?", "How is heart failure managed?", "Why can heart failure cause swollen ankles?", "What does reduced pumping function mean?", "Can heart failure cause breathlessness?", "What should caregivers know about heart failure?"],
"tests": ["What is an electrocardiogram?", "Why might a doctor order an echocardiogram?", "What happens during a stress test?", "How does a Holter monitor work?", "What can an ECG show?", "Do heart tests hurt?", "What is an ultrasound of the heart?", "Why do doctors check cardiac blood markers?"],
"medications": ["What are beta blockers used for?", "Why might someone take a statin?", "What should I ask about heart medicine side effects?", "Can I stop my prescribed heart medication?", "What is an anticoagulant?", "Why are ACE inhibitors prescribed?", "How should I remember my heart medicines?", "Can heart medicines interact with other drugs?"]
}
texts = [q for intent, qs in EXAMPLES.items() for q in qs]
labels = [intent for intent, qs in EXAMPLES.items() for q in qs]
assert len(texts) >= 40 and len(set(labels)) >= 5
print(f"{len(texts)} labeled queries across {len(set(labels))} intents")


## 3. Domain concepts and relationships

The knowledge graph below has over eight concepts and ten directed relationships. Relationships are educational associations, not a patient-specific causal model.

In [ ]:
CONCEPTS = {
 "Heart": "organ that circulates blood", "Coronary artery disease": "narrowing of arteries supplying the heart",
 "Blood pressure": "force of blood against artery walls", "Cholesterol": "blood lipid relevant to cardiovascular risk",
 "Smoking": "tobacco exposure that harms cardiovascular health", "Physical activity": "movement that supports cardiovascular fitness",
 "Diet": "food pattern influencing cardiovascular risk", "ECG": "recording of the heart's electrical activity",
 "Echocardiogram": "ultrasound imaging of heart structure and function", "Palpitations": "awareness of heartbeat, often fast or irregular",
 "Heart failure": "condition where heart function is inadequate for body needs", "Medication": "prescribed therapy used to manage a condition"}
RELATIONSHIPS = [
 ("High blood pressure", "increases risk of", "Coronary artery disease"),
 ("Smoking", "increases risk of", "Coronary artery disease"),
 ("High cholesterol", "increases risk of", "Coronary artery disease"),
 ("Physical activity", "supports", "Heart health"),
 ("Heart-healthy diet", "supports", "Heart health"),
 ("ECG", "records electrical activity of", "Heart"),
 ("Echocardiogram", "images structure and function of", "Heart"),
 ("Heart failure", "may cause", "Breathlessness"),
 ("Heart failure", "may cause", "Swelling"),
 ("Medication", "may be prescribed to manage", "High blood pressure"),
 ("Medication", "may be prescribed to manage", "Heart failure"),
 ("Palpitations", "may be assessed with", "ECG"),
]
print(f"{len(CONCEPTS)} concepts; {len(RELATIONSHIPS)} relationships")


## 4. NLP preprocessing, TF-IDF and intent model

Preprocessing is explicit: lowercase conversion, tokenization, stopword removal, and lemmatization. TF-IDF features feed a Multinomial Naive Bayes classifier. The held-out split is stratified; the small dataset means metrics are illustrative, not evidence of clinical readiness.

In [ ]:
lemmatizer = WordNetLemmatizer()
STOPWORDS = set(ENGLISH_STOP_WORDS)
def preprocess(text):
    text = text.lower()
    tokens = re.findall(r"[a-z]+", text)  # tokenization
    # Normalize a few common lay expressions used in the example test set.
    synonyms = {"ticker": "heart", "flutters": "fluttering", "skips": "skipped"}
    tokens = [synonyms.get(t, t) for t in tokens]
    tokens = [t for t in tokens if t not in STOPWORDS]  # stopword removal
    return " ".join(lemmatizer.lemmatize(t) for t in tokens)  # lemmatization

X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.25, random_state=42, stratify=labels)
model = Pipeline([
    ("tfidf", TfidfVectorizer(preprocessor=preprocess, tokenizer=str.split,
                              token_pattern=None, lowercase=False, ngram_range=(1,2))),
    ("classifier", MultinomialNB(alpha=0.25))
])
model.fit(X_train, y_train)
pred = model.predict(X_test)
print("Accuracy:", round(accuracy_score(y_test, pred), 3))
print("Precision (macro):", round(precision_score(y_test, pred, average="macro", zero_division=0), 3))
print("Recall (macro):", round(recall_score(y_test, pred, average="macro", zero_division=0), 3))
print("F1 (macro):", round(f1_score(y_test, pred, average="macro", zero_division=0), 3))
print(classification_report(y_test, pred, labels=INTENTS, zero_division=0))
cm = confusion_matrix(y_test, pred, labels=INTENTS)
ConfusionMatrixDisplay(cm, display_labels=INTENTS).plot(xticks_rotation=45, cmap="Blues")
plt.tight_layout(); plt.show()


## 5. Healthcare knowledge base

Each supported intent has a concise general-information response. Content is educational and should be reviewed by a qualified clinician before real-world use.

In [ ]:
KNOWLEDGE_BASE = {
"chest_pain": "Chest discomfort has many possible causes, including heart, lung, digestive, muscle, and anxiety-related causes. A clinician can assess symptoms. Sudden or severe symptoms require emergency help.",
"palpitations": "Palpitations are sensations of a racing, fluttering, pounding, or skipped heartbeat. Triggers can include stress, caffeine, medicines, or a rhythm problem. Recurrent symptoms merit clinical assessment.",
"risk_factors": "Heart disease risk can be affected by blood pressure, cholesterol, diabetes, tobacco use, age, family history, and activity level. A healthcare professional can assess personal risk.",
"prevention": "Heart-supportive habits include regular activity suited to your abilities, a balanced eating pattern, avoiding tobacco, adequate sleep, and managing blood pressure, cholesterol, and diabetes with professional guidance.",
"blood_pressure": "Blood pressure is recorded as systolic over diastolic pressure. A single reading does not establish a diagnosis; measurement technique and repeated readings matter. Discuss readings with a healthcare professional.",
"heart_failure": "Heart failure means the heart is not pumping or filling well enough for the body's needs; it does not mean the heart has stopped. Symptoms can include breathlessness, fatigue, or swelling. Evaluation and treatment are individualized.",
"tests": "Common heart tests include ECG, echocardiogram, rhythm monitors, stress testing, and selected blood tests. The clinician ordering a test can explain its purpose and preparation.",
"medications": "Heart medicines vary by condition and can have side effects or interactions. Take them as prescribed and ask a pharmacist or prescriber before changing or stopping a medicine. Seek urgent help for a severe allergic reaction."
}


## 6. Rule-based reasoning and integrated assistant

Rules: (1) emergency signs override classification, (2) low confidence asks for clarification, (3) unsupported topics are declined with a safe redirect, (4) symptom responses carry clinician guidance, and (5) medication changes are referred to the prescriber. Threshold is configurable and should be validated before deployment.

In [ ]:
EMERGENCY_PATTERNS = [
 r"chest pain", r"chest pressure", r"crushing (chest|pain)", r"shortness of breath",
 r"cannot breathe", r"can't breathe", r"faint(ed|ing)?", r"unconscious",
 r"face droop", r"weakness on one side", r"stroke symptoms"
]
SUPPORTED_TERMS = set("heart cardiac blood pressure hypertension cholesterol palpitations heartbeat chest ecg electrocardiogram echocardiogram heart failure medication medicine statin beta blocker risk exercise diet smoking artery ticker flutter flutters skips skipped".split())
CONFIDENCE_THRESHOLD = 0.28
def answer_query(query):
    q = query.strip()
    if not q: return "Please enter a question."
    low = q.lower()
    if any(re.search(p, low) for p in EMERGENCY_PATTERNS):
        return ("EMERGENCY: This may be a medical emergency. Call your local emergency number now or go to the nearest emergency department. "
                "Do not delay for an online response. If possible, have someone stay with you.")
    tokens = set(re.findall(r"[a-z]+", low))
    if not (tokens & SUPPORTED_TERMS):
        return "I can answer general heart-health questions in these areas: symptoms, risk and prevention, blood pressure, heart failure, tests, and medication education. This query appears outside that scope; please ask a heart-health question or contact a qualified professional."
    probs = model.predict_proba([q])[0]
    best_i = probs.argmax(); intent = model.classes_[best_i]; confidence = float(probs[best_i])
    if confidence < CONFIDENCE_THRESHOLD:
        return (f"I'm not confident I understood (confidence {confidence:.2f}). Could you rephrase or specify whether you mean symptoms, prevention, blood pressure, heart failure, tests, or medicines? "
                "For personal medical advice, contact a healthcare professional.")
    response = KNOWLEDGE_BASE[intent]
    if intent in ("chest_pain", "palpitations"):
        response += " If symptoms are severe, sudden, worsening, or accompanied by fainting or trouble breathing, seek urgent medical care."
    if intent == "medications":
        response += " Never stop or change a prescribed medicine based only on this assistant."
    return f"**{TAXONOMY[next(k for k,v in TAXONOMY.items() if intent in v)][intent]}** (intent: `{intent}`, confidence: {confidence:.2f})\n\n{response}\n\nEducational information only; this assistant does not diagnose."

# Quick illustrative safety checks (also available for user editing below)
REQUIRED_SCENARIOS = {
 "emergency": "I have crushing chest pressure and can't breathe",
 "unsupported": "How do I treat a broken ankle?",
 "low_confidence": "What is the heart blue spoon situation?",
 "reworded": "My ticker flutters and skips beats sometimes",
}
for name, query in REQUIRED_SCENARIOS.items():
    print(f"[{name}] {query}\n{answer_query(query)}\n")


## 7. Interactive multi-query interface

Enter one question per line (or separate with semicolons) and click **Get responses**. The interface accepts multiple questions in one submission.

In [ ]:
query_box = widgets.Textarea(
    value="What can I do to keep my heart healthy?\nWhy is my heartbeat fluttering?",
    placeholder="Enter one question per line", description="Queries:",
    layout=widgets.Layout(width="95%", height="120px"))
run_button = widgets.Button(description="Get responses", button_style="primary")
output = widgets.Output()
def _run(_):
    with output:
        output.clear_output()
        queries = [x.strip() for x in re.split(r"[\n;]+", query_box.value) if x.strip()]
        for i, q in enumerate(queries, 1):
            display(Markdown(f"### Query {i}: {q}\n\n{answer_query(q)}"))
run_button.on_click(_run)
display(query_box, run_button, output)


## Required test checklist

This notebook tests one query per defined intent through the model and evaluates the required variants. Edit wording to probe robustness. A small training corpus can produce low or misleading scores, and language coverage is limited. Do not use this prototype for diagnosis, triage decisions, or medication changes.

In [ ]:
TEST_QUERIES = {
 "chest_pain": "I have a question about chest tightness",
 "palpitations": "Why does my heartbeat flutter?",
 "risk_factors": "Does tobacco raise cardiovascular risk?",
 "prevention": "What habits help protect my heart?",
 "blood_pressure": "What do blood pressure readings tell me?",
 "heart_failure": "Can weak heart pumping cause swollen ankles?",
 "tests": "What does an ECG measure?",
 "medications": "Why are beta blockers prescribed?",
 "differently_worded": "My ticker flutters and skips beats sometimes",
 "unsupported": "How do I treat a broken ankle?",
 "low_confidence": "What is the heart blue spoon situation?",
 "emergency": "I have crushing chest pressure and can't breathe"
}
for name, query in TEST_QUERIES.items():
    print(f"\n--- {name}: {query} ---\n{answer_query(query)}")
